## loading SQL function

In [0]:
from pyspark.sql import functions as F


In [0]:
%sql
use catalog centuryhealthdatabase

## SYMPTOMS LONG FORMAT

In [0]:
symptoms = spark.table("silver.symptoms")

symptoms_long = symptoms.select(
    "patient",
    F.explode(F.split("symptoms", ";")).alias("pair")
).select(
    "patient",
    F.trim(F.split("pair", ":")[0]).alias("symptom_name"),
    F.split("pair", ":")[1].cast("int").alias("symptom_value")
)

symptoms_long.write.mode("overwrite").format("delta").saveAsTable("gold.symptoms_observation")


## Master Tble

In [0]:
patients = spark.table("silver.patients").alias("p")
encounters = spark.table("silver.encounters").alias("e")
conditions = spark.table("silver.conditions").alias("c")
medications = spark.table("silver.medications").alias("m")
symptoms_long = spark.table("gold.symptoms_observation").alias("s")

master = encounters \
.join(patients, F.col("e.patient") == F.col("p.patient_id"), "left") \
.join(conditions, F.col("e.id") == F.col("c.encounter"), "left") \
.join(medications, F.col("e.id") == F.col("m.encounter"), "left") \
.join(symptoms_long, F.col("e.patient") == F.col("s.patient"), "left") \
.select(
    F.col("e.id").alias("encounter_id"),
    F.col("e.patient"),
    F.col("e.start").alias("encounter_start"),
    F.col("e.stop").alias("encounter_stop"),
    F.col("e.encounterclass"),
    
    F.col("p.gender"),
    F.col("p.race"),
    F.col("p.ethnicity"),
    F.col("p.birthdate"),

    F.col("c.code").alias("condition_code"),
    F.col("c.description").alias("condition_name"),

    F.col("m.code").alias("medication_code"),
    F.col("m.description").alias("medication_name"),
    F.col("m.totalcost"),

    F.col("s.symptom_name"),
    F.col("s.symptom_value")
)

master.write.mode("overwrite").format("delta").saveAsTable("gold.master_health")

print("Gold Layer Complete")

Gold Layer Complete
